# Customer Behavior Data Pipeline
Cleans a raw customer shopping behavior dataset with pandas, engineers a few analysis-ready fields, and loads the result into PostgreSQL for SQL-based analysis.

## 1. Load the raw data

In [ ]:
import pandas as pd
import os

In [ ]:
df = pd.read_csv('data/customer_shopping_behavior.csv')

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 2. Check for missing values

In [ ]:
df.isnull().sum()

`Review Rating` has missing values. Fill them using the median rating **within each product category**, rather than a single global median, so the fill reflects category-level rating norms.

In [ ]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [ ]:
df.isnull().sum()  # confirm no missing values remain

## 3. Clean and standardize column names

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [ ]:
df.columns = df.columns.str.replace('purchase_amount_(usd)', 'purchase_amount')

In [ ]:
df.columns

## 4. Feature engineering

Bucket customers into quartile-based age groups for easier segmentation in downstream analysis.

In [ ]:
labels = ['Young Adult', 'Adult', 'Middle Aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)
df[['age', 'age_group']].head(10)

Convert the categorical `frequency_of_purchases` field into a numeric `purchase_frequency_days` field, which is far more useful for aggregation and comparison in SQL/BI tools.

In [ ]:
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)
df[['frequency_of_purchases', 'purchase_frequency_days']].head(10)

## 5. Drop redundant columns

`discount_applied` and `promo_code_used` turned out to be identical across every row, so the second column is redundant and safe to drop.

In [ ]:
(df['discount_applied'] == df['promo_code_used']).all()

In [ ]:
df = df.drop('promo_code_used', axis=1)

## 6. Load cleaned data into PostgreSQL
Credentials are read from environment variables rather than hardcoded, so this notebook is safe to share publicly. Set these in your own environment before running:
```
export DB_USERNAME=your_username
export DB_PASSWORD=your_password
export DB_HOST=localhost
export DB_PORT=5432
export DB_NAME=customer_database
```

In [ ]:
# !pip install "psycopg[binary]" sqlalchemy

In [ ]:
from sqlalchemy import create_engine
import os

username = os.environ['DB_USERNAME']
password = os.environ['DB_PASSWORD']
host = os.environ.get('DB_HOST', 'localhost')
port = os.environ.get('DB_PORT', '5432')
database = os.environ.get('DB_NAME', 'customer_database')

engine = create_engine(f"postgresql+psycopg://{username}:{password}@{host}:{port}/{database}")

In [ ]:
table_name = 'customer'
df.to_sql(table_name, engine, if_exists='replace', index=False)
print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

## 7. Export cleaned data
Also save a cleaned CSV copy, useful for the Power BI dashboard or any tool that doesn't connect directly to Postgres.

In [ ]:
df.to_csv('data/customer_shopping_behavior_cleaned.csv', index=False)